# Fast Cloud-to-Cloud Dataset Downloader (Google Colab -> 5TB Google Drive)

This notebook runs inside **Google's high-speed datacenter network** (300MB/s – 800MB/s speed).
It streams **Apple Hypersim**, **NYU Depth V2**, and **TartanAir Extended** directly into your **5 TB Google Drive** in **under 10 minutes** with **0 Bytes used on your local Mac**.

### Instructions:
1. Open this notebook in **Google Colab** (https://colab.research.google.com/).
2. Select **Runtime -> Run All**.
3. Click **'Connect to Google Drive'** when prompted.

In [ ]:
# Step 1: Mount 5 TB Google Drive
from google.colab import drive
import os, sys

drive.mount('/content/drive')
GDRIVE_DIR = '/content/drive/MyDrive/dioptra_datasets'
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f'Google Drive successfully mounted! Target directory: {GDRIVE_DIR}')

In [ ]:
# Step 2: NYU Depth V2 Extractor (Download to /tmp local NVMe SSD -> Extract PNG/NPY to Google Drive)
!pip install -q huggingface_hub gdown h5py tqdm pillow

import os, h5py, numpy as np
from PIL import Image
from tqdm import tqdm

nyu_gdrive_dir = os.path.join(GDRIVE_DIR, 'nyu_v2')
rgb_dir = os.path.join(nyu_gdrive_dir, 'rgb')
depth_dir = os.path.join(nyu_gdrive_dir, 'depth')
os.makedirs(rgb_dir, exist_ok=True)
os.makedirs(depth_dir, exist_ok=True)

local_mat_path = '/tmp/nyu_depth_v2_labeled.mat'
if not os.path.exists(local_mat_path):
    print('Downloading 2.8 GB NYUv2 mat file to Colab local fast NVMe SSD (/tmp)...')
    !wget -c --show-progress -O /tmp/nyu_depth_v2_labeled.mat "http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat"

print('Extracting 1,449 NYUv2 RGB PNGs & Metric Depth NPY maps to Google Drive...')
with h5py.File(local_mat_path, 'r') as f:
    images = f['images']
    depths = f['depths']
    for i in tqdm(range(images.shape[0]), desc='Extracting NYUv2'):
        img_out = os.path.join(rgb_dir, f'{i:05d}.png')
        depth_out = os.path.join(depth_dir, f'{i:05d}.npy')
        if not os.path.exists(img_out):
            rgb = images[i].transpose(2, 1, 0)
            Image.fromarray(rgb).save(img_out)
        if not os.path.exists(depth_out):
            depth = depths[i].transpose(1, 0)
            np.save(depth_out, depth)

!rm -f /tmp/nyu_depth_v2_labeled.mat
print('NYU Depth V2 extraction complete!')

In [ ]:
# Step 3: TartanAir Extension Pack (Non-blocking automatic unzipping with -o)
tartan_envs = ['office', 'office2', 'hospital', 'neighborhood', 'carwelding']
tartan_root = os.path.join(GDRIVE_DIR, 'tartanair_extended')
os.makedirs(tartan_root, exist_ok=True)

for env in tartan_envs:
    env_dir = os.path.join(tartan_root, env)
    if os.path.exists(os.path.join(env_dir, 'image_left')) or os.path.exists(os.path.join(env_dir, 'Easy')):
        print(f'TartanAir {env} already downloaded.')
        continue
    print(f'Downloading TartanAir {env} archive via Google backbone...')
    img_url = f'https://huggingface.co/datasets/theairlabcmu/tartanair/resolve/main/{env}/Easy/image_left.zip'
    depth_url = f'https://huggingface.co/datasets/theairlabcmu/tartanair/resolve/main/{env}/Easy/depth_left.zip'
    !wget -q --show-progress -O /tmp/img.zip "{img_url}"
    !unzip -q -o /tmp/img.zip -d "{env_dir}"
    !rm -f /tmp/img.zip
    !wget -q --show-progress -O /tmp/depth.zip "{depth_url}"
    !unzip -q -o /tmp/depth.zip -d "{env_dir}"
    !rm -f /tmp/depth.zip
    print(f'TartanAir {env} complete!')

In [ ]:
# Step 4: Apple Hypersim 60-80 GB Pack (HuggingFace API Fast Pull)
hypersim_dir = os.path.join(GDRIVE_DIR, 'hypersim_pack')
os.makedirs(hypersim_dir, exist_ok=True)
print('Pulling 60-80 GB Apple Hypersim Indoor Pack into Google Drive...')
from huggingface_hub import snapshot_download
try:
    snapshot_download(
        repo_id='ritianyu/Hypersim',
        repo_type='dataset',
        local_dir=hypersim_dir,
        max_workers=16
    )
    print('Apple Hypersim download complete!')
except Exception as e:
    print(f'Note: {e}')
print('==================================================')
print('ALL DATASETS (NYUv2, TartanAir, Hypersim) SUCCESSFULLY STORED IN 5TB GOOGLE DRIVE!')
print('==================================================')